# Check LSST colors of template SEDs

Made by Shenming Fu. 

Eric Charles pointed me to the h5 file (which could be made by Sam Schmidt).

First run `https://github.com/LSSTDESC/pz_data_challenge.git`

The data is in the `nb` folder.

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
FILENAME = "CWWSB_LSST_Roman_colors_YJH.hdf5"

BAND_LIST = [
'u',
'g',
'r',
'i',
'z',
'y',
]

In [ ]:
def get_redshift():
    with h5py.File(FILENAME, "r") as hf:
        redshift = hf["redshift"][:]
    return redshift

In [ ]:
def get_color(SED_name, band1_name, band2_name):
    
    band1_index = BAND_LIST.index(band1_name)
    band2_index = BAND_LIST.index(band2_name)

    color_name = f"{band1_name}{band2_name}"
    
    with h5py.File(FILENAME, "r") as hf:
        if (band1_index+1)==band2_index:
            color = hf[f"{SED_name}_{color_name}"][:]
        else:
            color = np.zeros_like(hf["redshift"][:])
            
            band1_name_tmp = band1_name
            while 1:
                band2_name_tmp = BAND_LIST[BAND_LIST.index(band1_name_tmp)+1]
                color_name_tmp = f"{band1_name_tmp}{band2_name_tmp}"
                color_tmp = hf[f"{SED_name}_{color_name_tmp}"][:]
                color += color_tmp
                band1_name_tmp = band2_name_tmp
                if band2_name_tmp == band2_name:
                    break
            
    return color

In [ ]:
def make_cc_plot2(ax, SED_name, band1_name, band2_name, band3_name, band4_name, color='C0'):

    alpha = 0.5
    redshift = get_redshift()
    # step: 0.01
    #print(f"max redshift: {np.max(redshift)}")
    sel = redshift <= 2.5
    
    color1 = get_color(SED_name, band1_name, band2_name)
    color2 = get_color(SED_name, band3_name, band4_name)
    
    ax.plot(color1[sel], color2[sel], label=SED_name, alpha=alpha)
    
    sel = redshift==0
    ax.plot(color1[sel], color2[sel], marker='o', color=color, alpha=alpha)

    sel = redshift==0.5
    ax.plot(color1[sel], color2[sel], marker='>', color=color, alpha=alpha)

    sel = redshift==1.
    ax.plot(color1[sel], color2[sel], marker='s', color=color, alpha=alpha)

    sel = redshift==1.5
    ax.plot(color1[sel], color2[sel], marker='*', color=color, alpha=alpha)

    sel = redshift==2.0
    ax.plot(color1[sel], color2[sel], marker='P', color=color, alpha=alpha)

    sel = redshift==2.5
    ax.plot(color1[sel], color2[sel], marker='X', color=color, alpha=alpha)
    
    return 0

In [ ]:
SED_list = [
"El_B2004a", 
#"Im_B2004a",
#"SB2_B2004a",
#"SB3_B2004a",
"Sbc_B2004a",
"Scd_B2004a",
#"ssp_5Myr_z008",
#"ssp_25Myr_z008",
]

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(5,5),layout='constrained')

band1_name, band2_name, band3_name, band4_name = 'r', 'z', 'g', 'i'

for ind, SED in enumerate(SED_list):
    make_cc_plot2(ax, SED, band1_name, band2_name, band3_name, band4_name, f'C{ind}')


color1_name = f"{band1_name} - {band2_name}"
color2_name = f"{band3_name} - {band4_name}"
ax.set_xlabel(color1_name)
ax.set_ylabel(color2_name)

ax.set_xlim([-2, 5])
ax.set_ylim([-2, 5])

ax.grid(ls=':')
ax.legend()

#plt.savefig("model_colors.png")
plt.savefig("model_colors_v2.pdf")